# Final 16-Dataset Query Evaluation Journal

This notebook is the final analysis journal for the LogLite compressed-query project. It is designed for the full complete-suite run:

- 16 LogHub datasets
- 8 locked query families
- 4 execution modes
- 1 warmup + 10 measured repetitions per cell

The notebook treats `query_eval` outputs as the source of truth. It does not execute queries or reimplement search logic; it only loads `manifest.json`, `raw_runs.jsonl`, and the aggregate CSVs produced by `python3 -m query_eval.cli run-suite`.

## 1. Setup and Result Directory

Set `RESULTS_DIR` manually if you want a specific run. If it is left as `None`, the notebook picks the newest result bundle whose manifest looks like a 16-dataset complete static evaluation.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from statistics import median

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 140)
plt.style.use('ggplot')

RESULTS_ROOT = Path('evaluation_results/query_eval')
RESULTS_DIR = None  # Example: Path('evaluation_results/query_eval/20260516_012345_complete_static_evaluation')
EXPORT_FIGURES = True
EXPORT_TABLES = True
LOAD_RAW_LEDGER = False  # Full raw JSONL is large; stream it later for failure samples.

MODE_ORDER = ['decompressed_text', 'full_decompression', 'minor_optimization', 'static_bloom']
MODE_LABELS = {
    'decompressed_text': 'Decompressed Text',
    'full_decompression': 'Full Decompression',
    'minor_optimization': 'Minor Optimization',
    'static_bloom': 'Static Bloom',
}
MODE_COLORS = {
    'decompressed_text': '#4C78A8',
    'full_decompression': '#F58518',
    'minor_optimization': '#54A24B',
    'static_bloom': '#B279A2',
}
QUERY_ORDER = [
    'common_token',
    'medium_token',
    'rare_token',
    'common_phrase',
    'selective_phrase',
    'numeric_identifier',
    'conjunctive',
    'bloom_stress_substring',
]
QUERY_LABELS = {
    'common_token': 'Common Token',
    'medium_token': 'Medium Token',
    'rare_token': 'Rare Token',
    'common_phrase': 'Common Phrase',
    'selective_phrase': 'Selective Phrase',
    'numeric_identifier': 'Numeric ID',
    'conjunctive': 'Conjunctive',
    'bloom_stress_substring': 'Stress Substring',
}


def resolve_results_dir(results_dir: Path | None = None) -> Path:
    if results_dir is not None:
        if not results_dir.exists():
            raise FileNotFoundError(f'Requested results directory does not exist: {results_dir}')
        return results_dir
    if not RESULTS_ROOT.exists():
        raise FileNotFoundError('No evaluation_results/query_eval directory found. Run the complete suite first.')

    candidates = []
    for path in sorted(RESULTS_ROOT.iterdir()):
        manifest_path = path / 'manifest.json'
        if not path.is_dir() or not manifest_path.exists():
            continue
        try:
            manifest = json.loads(manifest_path.read_text())
        except json.JSONDecodeError:
            continue
        datasets = manifest.get('datasets', [])
        queries = manifest.get('queries', [])
        modes = manifest.get('modes', [])
        label = manifest.get('run_config', {}).get('config_label', '')
        if len(datasets) == 16 and len(queries) == 8 and len(modes) == 4 and 'complete' in label:
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(
            'No complete 16-dataset result bundle found. Run:\n'
            'python3 -m query_eval.cli run-suite --profile complete_static_evaluation '
            '--repetitions 10 --warmups 1 --config-label complete_static_evaluation '
            '--config-version complete_static.v1'
        )
    return candidates[-1]


def order_modes(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    if 'mode_name' in output.columns:
        output['mode_name'] = pd.Categorical(output['mode_name'], MODE_ORDER, ordered=True)
    if 'query_id' in output.columns:
        output['query_id'] = pd.Categorical(output['query_id'], QUERY_ORDER, ordered=True)
    sort_columns = [column for column in ['dataset_slug', 'query_id', 'mode_name'] if column in output.columns]
    if sort_columns:
        output = output.sort_values(sort_columns).reset_index(drop=True)
    return output


def export_dir(results_dir: Path) -> Path:
    output = results_dir / 'final_16_dataset_journal_exports'
    output.mkdir(parents=True, exist_ok=True)
    return output


def maybe_savefig(fig: plt.Figure, results_dir: Path, stem: str) -> None:
    if not EXPORT_FIGURES:
        return
    output_path = export_dir(results_dir) / f'{stem}.png'
    fig.savefig(output_path, dpi=220, bbox_inches='tight')
    print(f'Saved figure: {output_path}')


def maybe_savetable(frame: pd.DataFrame, results_dir: Path, stem: str) -> None:
    if not EXPORT_TABLES:
        return
    output_path = export_dir(results_dir) / f'{stem}.csv'
    frame.to_csv(output_path, index=False)
    print(f'Saved table: {output_path}')


def plot_heatmap(frame: pd.DataFrame, title: str, cmap: str = 'viridis', vmin=None, vmax=None, fmt: str = '.2f', figsize=(12, 7)):
    fig, ax = plt.subplots(figsize=figsize)
    image = ax.imshow(frame.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_title(title)
    ax.set_xticks(range(len(frame.columns)))
    ax.set_xticklabels([QUERY_LABELS.get(str(col), str(col)) for col in frame.columns], rotation=35, ha='right')
    ax.set_yticks(range(len(frame.index)))
    ax.set_yticklabels(frame.index)
    for y in range(frame.shape[0]):
        for x in range(frame.shape[1]):
            value = frame.iloc[y, x]
            if pd.notna(value):
                ax.text(x, y, format(value, fmt), ha='center', va='center', fontsize=8, color='white' if value < 0.5 else 'black')
    fig.colorbar(image, ax=ax, fraction=0.035, pad=0.03)
    fig.tight_layout()
    return fig, ax

results_dir = resolve_results_dir(RESULTS_DIR)
print('Using results directory:', results_dir)

## 2. Load the Complete Result Bundle

This section loads the aggregate outputs and checks that the bundle has the expected complete-suite shape. The raw JSONL is intentionally optional because it can be large; aggregate CSVs are enough for most plots.

In [ ]:
manifest = json.loads((results_dir / 'manifest.json').read_text())
cell_df = order_modes(pd.read_csv(results_dir / 'cell_level_aggregate.csv'))
query_df = order_modes(pd.read_csv(results_dir / 'query_level_aggregate.csv'))
dataset_df = order_modes(pd.read_csv(results_dir / 'dataset_level_aggregate.csv'))
suite_df = order_modes(pd.read_csv(results_dir / 'suite_summary.csv'))
static_bloom_df = pd.read_csv(results_dir / 'static_bloom_summary.csv')
complete_summary_df = pd.read_csv(results_dir / 'complete_evaluation_summary.csv')
query_manifest_df = pd.read_csv(results_dir / 'query_manifest.csv')
dataset_coverage_df = pd.read_csv(results_dir / 'dataset_coverage.csv')
raw_jsonl_path = results_dir / 'raw_runs.jsonl'
raw_record_count = sum(1 for _ in raw_jsonl_path.open('r', encoding='utf-8')) if raw_jsonl_path.exists() else 0

expected_cells = len(manifest['datasets']) * len(manifest['queries']) * len(manifest['modes'])
expected_raw_records = expected_cells * (manifest['run_config']['warmups'] + manifest['run_config']['repetitions'])

print('Run created at:', manifest['created_at'])
print('Config:', manifest['run_config']['config_label'], manifest['run_config']['config_version'])
print('Datasets:', len(manifest['datasets']), manifest['datasets'])
print('Queries:', len(manifest['queries']), manifest['queries'])
print('Modes:', len(manifest['modes']), manifest['modes'])
print('Expected cells:', expected_cells, '| observed cell rows:', len(cell_df))
print('Expected raw records:', expected_raw_records, '| observed raw records:', raw_record_count)
print('Static Bloom rows:', len(static_bloom_df))

assert len(manifest['datasets']) == 16
assert len(manifest['queries']) == 8
assert len(manifest['modes']) == 4
assert len(cell_df) == expected_cells
assert raw_record_count == expected_raw_records

if LOAD_RAW_LEDGER:
    raw_df = pd.read_json(raw_jsonl_path, lines=True)
    print('Loaded raw ledger into memory:', raw_df.shape)
else:
    raw_df = None

print('\nRun config:')
display(pd.DataFrame([manifest['run_config']]))

## 3. Dataset and Query Coverage

The suite covers all 16 registered LogHub TEXT datasets. The AL and NDL columns are the paper-style metadata used to reason about line length diversity and compression/query difficulty.

In [ ]:
coverage_display = dataset_coverage_df.sort_values('dataset_slug').reset_index(drop=True)
display(coverage_display)
maybe_savetable(coverage_display, results_dir, 'dataset_coverage')

query_display = query_manifest_df[['query_id', 'family', 'token_safe', 'is_stress_query', 'expected_selectivity_band']].drop_duplicates().reset_index(drop=True)
query_display['query_id'] = pd.Categorical(query_display['query_id'], QUERY_ORDER, ordered=True)
query_display = query_display.sort_values('query_id').reset_index(drop=True)
display(query_display)
maybe_savetable(query_display, results_dir, 'query_family_manifest')

## 4. Suite-Level Results

The suite summary compares the four execution modes. `decompressed_text` is the correctness baseline, `full_decompression` is the exact compressed-bitstream reference, `minor_optimization` is the earlier length/window filter, and `static_bloom` is the current query-aware static-window candidate.

In [ ]:
suite_display_df = suite_df.copy()
full_wall = float(suite_display_df.loc[suite_display_df['mode_name'] == 'full_decompression', 'median_wall_time_ms'].iloc[0])
text_wall = float(suite_display_df.loc[suite_display_df['mode_name'] == 'decompressed_text', 'median_wall_time_ms'].iloc[0])
suite_display_df['mode_label'] = suite_display_df['mode_name'].astype(str).map(MODE_LABELS)
suite_display_df['speedup_vs_full_decompression'] = full_wall / suite_display_df['median_wall_time_ms']
suite_display_df['wall_time_vs_decompressed_text'] = suite_display_df['median_wall_time_ms'] / text_wall
suite_display_df = suite_display_df[[
    'mode_name', 'mode_label', 'cell_count', 'median_wall_time_ms', 'median_cpu_time_ms',
    'median_peak_rss_mb', 'median_precision', 'median_recall', 'median_f1',
    'speedup_vs_full_decompression', 'wall_time_vs_decompressed_text', 'median_bloom_skip_rate',
    'all_cells_exact_set_match',
]]

display(suite_display_df.round(4))
maybe_savetable(suite_display_df, results_dir, 'suite_summary_enriched')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
plots = [
    ('median_wall_time_ms', 'Median Wall Time', 'ms'),
    ('median_cpu_time_ms', 'Median CPU Time', 'ms'),
    ('median_peak_rss_mb', 'Median Peak RSS', 'MB'),
    ('speedup_vs_full_decompression', 'Speedup vs Full Decompression', 'x'),
]
for ax, (column, title, ylabel) in zip(axes.flat, plots):
    plot_df = suite_display_df.sort_values('mode_name')
    ax.bar(plot_df['mode_label'], plot_df[column], color=[MODE_COLORS[str(mode)] for mode in plot_df['mode_name']])
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=20)
    if column == 'speedup_vs_full_decompression':
        ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
    for patch in ax.patches:
        value = patch.get_height()
        ax.annotate(f'{value:.2f}', (patch.get_x() + patch.get_width() / 2, value), xytext=(0, 4), textcoords='offset points', ha='center', fontsize=9)
fig.suptitle('Complete Suite Mode Comparison', fontsize=14)
fig.tight_layout()
maybe_savefig(fig, results_dir, 'suite_mode_comparison')
plt.show()

## 5. Correctness Analysis

Correctness is set equivalence against `decompressed_text`. The most important checks are whether `full_decompression` remains exact and where the two optimization modes lose recall.

In [ ]:
mode_correctness_df = (
    cell_df.groupby('mode_name', observed=False)
    .agg(
        cells=('mode_name', 'size'),
        exact_cells=('all_runs_exact_set_match', 'sum'),
        non_exact_cells=('all_runs_exact_set_match', lambda values: (~values).sum()),
        median_precision=('median_precision', 'median'),
        median_recall=('median_recall', 'median'),
        min_recall=('median_recall', 'min'),
        median_f1=('median_f1', 'median'),
        min_f1=('median_f1', 'min'),
        total_median_fp=('median_fp', 'sum'),
        total_median_fn=('median_fn', 'sum'),
    )
    .reset_index()
)
mode_correctness_df['mode_label'] = mode_correctness_df['mode_name'].astype(str).map(MODE_LABELS)
mode_correctness_df = mode_correctness_df[['mode_name', 'mode_label', 'cells', 'exact_cells', 'non_exact_cells', 'median_precision', 'median_recall', 'min_recall', 'median_f1', 'min_f1', 'total_median_fp', 'total_median_fn']]
display(mode_correctness_df.round(4))
maybe_savetable(mode_correctness_df, results_dir, 'mode_correctness_summary')

non_exact_cells_df = cell_df.loc[~cell_df['all_runs_exact_set_match']].copy()
non_exact_cells_df = non_exact_cells_df.sort_values(['mode_name', 'median_recall', 'median_f1', 'dataset_slug', 'query_id']).reset_index(drop=True)
print('Non-exact cells:', len(non_exact_cells_df))
display(non_exact_cells_df[[
    'dataset_slug', 'query_id', 'mode_name', 'median_match_count', 'median_fp', 'median_fn',
    'median_precision', 'median_recall', 'median_f1', 'median_bloom_skip_rate',
]].round(4))
maybe_savetable(non_exact_cells_df, results_dir, 'non_exact_cells')

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = mode_correctness_df.sort_values('mode_name')
ax.bar(plot_df['mode_label'], plot_df['exact_cells'], color=[MODE_COLORS[str(mode)] for mode in plot_df['mode_name']])
ax.set_title('Exact Cells by Mode')
ax.set_ylabel('Exact cell count')
ax.set_ylim(0, max(plot_df['cells']) + 5)
for _, row in plot_df.iterrows():
    x = list(plot_df['mode_label']).index(row['mode_label'])
    ax.annotate(f"{int(row['exact_cells'])}/{int(row['cells'])}", (x, row['exact_cells']), xytext=(0, 4), textcoords='offset points', ha='center')
fig.tight_layout()
maybe_savefig(fig, results_dir, 'exact_cells_by_mode')
plt.show()

## 6. Token-Safe vs Stress-Query Breakdown

The complete suite deliberately separates token-compatible queries from `bloom_stress_substring`, which is outside the static Bloom tokenizer assumption. Use this split in the report narrative.

In [ ]:
display(complete_summary_df.round(4))
maybe_savetable(complete_summary_df, results_dir, 'complete_evaluation_summary')

summary_pivot = complete_summary_df.pivot(index='mode_name', columns='query_group', values='exact_cell_count').reindex(MODE_ORDER)
fig, ax = plt.subplots(figsize=(11, 6))
summary_pivot.plot(kind='bar', ax=ax, color=['#4C78A8', '#72B7B2', '#E45756'])
ax.set_title('Exact Cell Counts by Query Group')
ax.set_xlabel('Mode')
ax.set_ylabel('Exact cell count')
ax.set_xticklabels([MODE_LABELS.get(str(label), str(label)) for label in summary_pivot.index], rotation=20)
ax.legend(title='Query group')
fig.tight_layout()
maybe_savefig(fig, results_dir, 'exact_cells_by_query_group')
plt.show()

## 7. Dataset-Level and Query-Level Performance

These views show where runtime varies by dataset and query family. This matters because query selectivity and line structure both influence how much work the compressed-domain modes can skip.

In [ ]:
runtime_by_dataset = dataset_df.pivot(index='dataset_slug', columns='mode_name', values='median_wall_time_ms').reindex(columns=MODE_ORDER)
runtime_by_dataset.columns = [MODE_LABELS[str(column)] for column in runtime_by_dataset.columns]
runtime_by_query = query_df.pivot(index='query_id', columns='mode_name', values='median_wall_time_ms').reindex(index=QUERY_ORDER, columns=MODE_ORDER)
runtime_by_query.index = [QUERY_LABELS.get(str(index), str(index)) for index in runtime_by_query.index]
runtime_by_query.columns = [MODE_LABELS[str(column)] for column in runtime_by_query.columns]

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
runtime_by_dataset.plot(kind='bar', ax=axes[0], color=[MODE_COLORS[mode] for mode in MODE_ORDER])
axes[0].set_title('Dataset-Level Median Wall Time')
axes[0].set_xlabel('Dataset')
axes[0].set_ylabel('ms')
axes[0].tick_params(axis='x', rotation=35)
axes[0].legend(title='Mode')

runtime_by_query.plot(kind='bar', ax=axes[1], color=[MODE_COLORS[mode] for mode in MODE_ORDER])
axes[1].set_title('Query-Level Median Wall Time')
axes[1].set_xlabel('Query family')
axes[1].set_ylabel('ms')
axes[1].tick_params(axis='x', rotation=35)
axes[1].legend(title='Mode')
fig.tight_layout()
maybe_savefig(fig, results_dir, 'dataset_query_runtime_breakdown')
plt.show()

maybe_savetable(runtime_by_dataset.reset_index(), results_dir, 'runtime_by_dataset')
maybe_savetable(runtime_by_query.reset_index().rename(columns={'index': 'query_family'}), results_dir, 'runtime_by_query')

## 8. Minor Optimization Failure Profile

The older `minor_optimization` mode is expected to lose recall because it skips records without maintaining the dynamic L-window state. This section measures that weakness rather than hiding it.

In [ ]:
minor_df = cell_df.loc[cell_df['mode_name'] == 'minor_optimization'].copy()
minor_recall = minor_df.pivot(index='dataset_slug', columns='query_id', values='median_recall').reindex(columns=QUERY_ORDER)
minor_f1 = minor_df.pivot(index='dataset_slug', columns='query_id', values='median_f1').reindex(columns=QUERY_ORDER)

fig, _ = plot_heatmap(minor_recall, 'Minor Optimization Recall Heatmap', cmap='RdYlGn', vmin=0, vmax=1, fmt='.2f', figsize=(14, 8))
maybe_savefig(fig, results_dir, 'minor_optimization_recall_heatmap')
plt.show()

fig, _ = plot_heatmap(minor_f1, 'Minor Optimization F1 Heatmap', cmap='RdYlGn', vmin=0, vmax=1, fmt='.2f', figsize=(14, 8))
maybe_savefig(fig, results_dir, 'minor_optimization_f1_heatmap')
plt.show()

full_rows = cell_df.loc[cell_df['mode_name'] == 'full_decompression', ['dataset_slug', 'query_id', 'median_wall_time_ms']].rename(columns={'median_wall_time_ms': 'full_wall_ms'})
minor_compare_df = minor_df.merge(full_rows, on=['dataset_slug', 'query_id'], how='left')
minor_compare_df['speedup_vs_full'] = minor_compare_df['full_wall_ms'] / minor_compare_df['median_wall_time_ms']
minor_compare_df['dataset_query'] = minor_compare_df['dataset_slug'] + ' / ' + minor_compare_df['query_id'].astype(str).map(QUERY_LABELS)

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(minor_compare_df['speedup_vs_full'], minor_compare_df['median_recall'], s=85, color=MODE_COLORS['minor_optimization'], alpha=0.85)
for _, row in minor_compare_df.iterrows():
    if row['median_recall'] < 1 or row['speedup_vs_full'] > minor_compare_df['speedup_vs_full'].quantile(0.85):
        ax.annotate(row['dataset_query'], (row['speedup_vs_full'], row['median_recall']), xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1)
ax.set_title('Minor Optimization: Speedup vs Recall')
ax.set_xlabel('Speedup vs Full Decompression (x)')
ax.set_ylabel('Recall')
ax.set_ylim(0, 1.05)
fig.tight_layout()
maybe_savefig(fig, results_dir, 'minor_speedup_vs_recall')
plt.show()

minor_failure_table = minor_compare_df.loc[minor_compare_df['median_fn'] > 0].sort_values(['median_recall', 'dataset_slug', 'query_id'])
display(minor_failure_table[[
    'dataset_slug', 'query_id', 'median_match_count', 'median_fn', 'median_recall', 'median_f1', 'speedup_vs_full'
]].round(4))
maybe_savetable(minor_failure_table, results_dir, 'minor_optimization_failure_cells')

## 9. Static Bloom Analysis

Static Bloom is the main current optimization candidate. It stores a 64-bit per-record token bitmap and reconstructs only candidate records that pass the bitmap test. Exact substring verification still happens after reconstruction, so observed false positives would be filtered before output; false negatives reveal query-token or codec metadata limitations.

In [ ]:
static_display = static_bloom_df.sort_values(['is_stress_query', 'token_safe', 'dataset_slug', 'query_id']).reset_index(drop=True)
display(static_display[[
    'dataset_slug', 'query_id', 'token_safe', 'is_stress_query', 'expected_selectivity_band',
    'baseline_match_count', 'static_bloom_match_count', 'false_positives', 'false_negatives',
    'precision', 'recall', 'f1', 'speedup_vs_full_decompression', 'bloom_skip_rate', 'exact_set_match'
]].round(4))
maybe_savetable(static_display, results_dir, 'static_bloom_cell_summary')

static_mode_summary = pd.DataFrame({
    'metric': [
        'cells', 'exact_cells', 'token_safe_cells', 'token_safe_exact_cells', 'stress_cells',
        'stress_exact_cells', 'total_false_positives', 'total_false_negatives', 'median_skip_rate',
        'median_speedup_vs_full', 'median_recall', 'median_f1',
    ],
    'value': [
        len(static_bloom_df),
        int(static_bloom_df['exact_set_match'].sum()),
        int(static_bloom_df['token_safe'].sum()),
        int(static_bloom_df.loc[static_bloom_df['token_safe'], 'exact_set_match'].sum()),
        int(static_bloom_df['is_stress_query'].sum()),
        int(static_bloom_df.loc[static_bloom_df['is_stress_query'], 'exact_set_match'].sum()),
        int(static_bloom_df['false_positives'].sum()),
        int(static_bloom_df['false_negatives'].sum()),
        static_bloom_df['bloom_skip_rate'].median(),
        static_bloom_df['speedup_vs_full_decompression'].median(),
        static_bloom_df['recall'].median(),
        static_bloom_df['f1'].median(),
    ],
})
display(static_mode_summary)
maybe_savetable(static_mode_summary, results_dir, 'static_bloom_summary_metrics')

In [ ]:
static_skip = static_bloom_df.pivot(index='dataset_slug', columns='query_id', values='bloom_skip_rate').reindex(columns=QUERY_ORDER)
static_recall = static_bloom_df.pivot(index='dataset_slug', columns='query_id', values='recall').reindex(columns=QUERY_ORDER)
static_speedup = static_bloom_df.pivot(index='dataset_slug', columns='query_id', values='speedup_vs_full_decompression').reindex(columns=QUERY_ORDER)

fig, _ = plot_heatmap(static_skip, 'Static Bloom Skip Rate Heatmap', cmap='viridis', vmin=0, vmax=1, fmt='.2f', figsize=(14, 8))
maybe_savefig(fig, results_dir, 'static_bloom_skip_rate_heatmap')
plt.show()

fig, _ = plot_heatmap(static_recall, 'Static Bloom Recall Heatmap', cmap='RdYlGn', vmin=0, vmax=1, fmt='.2f', figsize=(14, 8))
maybe_savefig(fig, results_dir, 'static_bloom_recall_heatmap')
plt.show()

fig, _ = plot_heatmap(static_speedup, 'Static Bloom Speedup vs Full Decompression', cmap='YlGnBu', vmin=0, vmax=max(1.0, static_speedup.max().max()), fmt='.1f', figsize=(14, 8))
maybe_savefig(fig, results_dir, 'static_bloom_speedup_heatmap')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = static_bloom_df['is_stress_query'].map({True: '#E45756', False: MODE_COLORS['static_bloom']})
axes[0].scatter(static_bloom_df['bloom_skip_rate'], static_bloom_df['recall'], s=80, c=colors, alpha=0.85)
axes[0].set_title('Static Bloom: Skip Rate vs Recall')
axes[0].set_xlabel('Bloom skip rate')
axes[0].set_ylabel('Recall')
axes[0].set_ylim(0, 1.05)
axes[0].axhline(1.0, color='black', linestyle='--', linewidth=1)

axes[1].scatter(static_bloom_df['bloom_skip_rate'], static_bloom_df['speedup_vs_full_decompression'], s=80, c=colors, alpha=0.85)
axes[1].set_title('Static Bloom: Skip Rate vs Speedup')
axes[1].set_xlabel('Bloom skip rate')
axes[1].set_ylabel('Speedup vs full decompression (x)')
axes[1].axhline(1.0, color='black', linestyle='--', linewidth=1)

for ax in axes:
    for _, row in static_bloom_df.iterrows():
        if row['false_negatives'] > 0 or row['speedup_vs_full_decompression'] > static_bloom_df['speedup_vs_full_decompression'].quantile(0.90):
            y_value = row['recall'] if ax is axes[0] else row['speedup_vs_full_decompression']
            ax.annotate(f"{row['dataset_slug']} / {QUERY_LABELS.get(row['query_id'], row['query_id'])}", (row['bloom_skip_rate'], y_value), xytext=(4, 4), textcoords='offset points', fontsize=8)

fig.tight_layout()
maybe_savefig(fig, results_dir, 'static_bloom_tradeoffs')
plt.show()

## 10. Failure Audit from Raw Runs

This cell streams `raw_runs.jsonl` and extracts representative false positives/false negatives for non-exact measured runs. It avoids loading the full raw ledger into memory.

In [ ]:
def stream_error_samples(raw_path: Path, mode_filter: str | None = None, limit: int = 80) -> pd.DataFrame:
    rows = []
    with raw_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            record = json.loads(line)
            if record.get('is_warmup'):
                continue
            if mode_filter is not None and record.get('mode_name') != mode_filter:
                continue
            correctness = record.get('correctness', {})
            if correctness.get('exact_set_match'):
                continue
            rows.append({
                'dataset_slug': record['dataset_slug'],
                'query_id': record['query_id'],
                'mode_name': record['mode_name'],
                'repetition_index': record['repetition_index'],
                'fp': correctness.get('fp'),
                'fn': correctness.get('fn'),
                'precision': correctness.get('precision'),
                'recall': correctness.get('recall'),
                'sample_false_positive': (record.get('sampled_false_positives') or [''])[0],
                'sample_false_negative': (record.get('sampled_false_negatives') or [''])[0],
            })
            if len(rows) >= limit:
                break
    return pd.DataFrame(rows)

minor_error_samples = stream_error_samples(raw_jsonl_path, mode_filter='minor_optimization')
static_error_samples = stream_error_samples(raw_jsonl_path, mode_filter='static_bloom')

print('Minor optimization sample errors:')
display(minor_error_samples)
maybe_savetable(minor_error_samples, results_dir, 'minor_error_samples')

print('Static Bloom sample errors:')
display(static_error_samples)
maybe_savetable(static_error_samples, results_dir, 'static_bloom_error_samples')

## 11. Final Tables for Report Writing

These compact tables are intended for direct use in the final report or presentation.

In [ ]:
report_mode_table = suite_display_df[[
    'mode_label', 'cell_count', 'median_wall_time_ms', 'median_cpu_time_ms', 'median_peak_rss_mb',
    'median_recall', 'median_f1', 'speedup_vs_full_decompression', 'median_bloom_skip_rate',
    'all_cells_exact_set_match',
]].round(4)
display(report_mode_table)
maybe_savetable(report_mode_table, results_dir, 'report_mode_table')

report_static_group_table = (
    static_bloom_df.assign(query_group=static_bloom_df['is_stress_query'].map({True: 'stress', False: 'token_safe'}))
    .groupby('query_group')
    .agg(
        cells=('query_id', 'size'),
        exact_cells=('exact_set_match', 'sum'),
        false_positives=('false_positives', 'sum'),
        false_negatives=('false_negatives', 'sum'),
        median_recall=('recall', 'median'),
        median_f1=('f1', 'median'),
        median_skip_rate=('bloom_skip_rate', 'median'),
        median_speedup=('speedup_vs_full_decompression', 'median'),
    )
    .reset_index()
)
display(report_static_group_table.round(4))
maybe_savetable(report_static_group_table, results_dir, 'report_static_bloom_group_table')

worst_static_cells = static_bloom_df.sort_values(['recall', 'f1', 'false_negatives'], ascending=[True, True, False]).head(20)
display(worst_static_cells[[
    'dataset_slug', 'query_id', 'token_safe', 'is_stress_query', 'baseline_match_count',
    'static_bloom_match_count', 'false_negatives', 'recall', 'f1', 'bloom_skip_rate',
    'speedup_vs_full_decompression', 'exact_set_match'
]].round(4))
maybe_savetable(worst_static_cells, results_dir, 'worst_static_bloom_cells')

## 12. Narrative Takeaways

Use the generated numbers above to write the final narrative carefully:

1. `full_decompression` validates the Python compressed-bitstream mirror when it remains exact against `decompressed_text`.
2. `minor_optimization` is a useful negative/transition result: it is faster but can lose recall because skipping breaks dynamic L-window state.
3. `static_bloom` is the main query-aware candidate: it can skip many records using per-record token metadata and then exact-check decoded candidates.
4. Stress substring queries should be interpreted separately because the Bloom metadata is token-based, not a general substring index.
5. Do not claim exact arbitrary substring search unless a fallback path or richer metadata is added.

In [ ]:
full_exact = bool(mode_correctness_df.loc[mode_correctness_df['mode_name'] == 'full_decompression', 'non_exact_cells'].iloc[0] == 0)
minor_exact = int(mode_correctness_df.loc[mode_correctness_df['mode_name'] == 'minor_optimization', 'exact_cells'].iloc[0])
minor_cells = int(mode_correctness_df.loc[mode_correctness_df['mode_name'] == 'minor_optimization', 'cells'].iloc[0])
static_exact = int(static_bloom_df['exact_set_match'].sum())
static_cells = len(static_bloom_df)
static_token = static_bloom_df.loc[static_bloom_df['token_safe']]
static_stress = static_bloom_df.loc[static_bloom_df['is_stress_query']]

print('Final narrative facts:')
print(f'- Full decompression exact: {full_exact}')
print(f'- Minor optimization exact cells: {minor_exact}/{minor_cells}')
print(f'- Static Bloom exact cells: {static_exact}/{static_cells}')
print(f'- Static Bloom token-safe exact cells: {int(static_token["exact_set_match"].sum())}/{len(static_token)}')
print(f'- Static Bloom stress-query exact cells: {int(static_stress["exact_set_match"].sum())}/{len(static_stress)}')
print(f'- Static Bloom total FP: {int(static_bloom_df["false_positives"].sum())}')
print(f'- Static Bloom total FN: {int(static_bloom_df["false_negatives"].sum())}')
print(f'- Static Bloom median skip rate: {static_bloom_df["bloom_skip_rate"].median():.2%}')
print(f'- Static Bloom median speedup vs full decompression: {static_bloom_df["speedup_vs_full_decompression"].median():.2f}x')
print('\nExports written under:', export_dir(results_dir))